In [1]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJECT_ROOT)

print("Project root added:", PROJECT_ROOT)


Project root added: c:\Users\SANIKA CHAUDHARI\OneDrive\Documents\GitHub\BE_project


In [29]:
from preprocessing.preprocess import run_preprocessing_media
import numpy as np 
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset , Dataset

In [30]:
class FrequencyDataset(Dataset):
    def __init__(self, freq_np, labels_np):
        self.x = torch.tensor(freq_np, dtype=torch.float32).permute(0, 3, 1, 2)  # (N,1,224,224)
        self.y = torch.tensor(labels_np, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


In [31]:
media_path = r"C:\Users\SANIKA CHAUDHARI\OneDrive\Documents\GitHub\BE_project\image.jpg"  # change path
spatial_np, freq_np, meta = run_preprocessing_media(media_path)

if freq_np is None:
    raise ValueError("No face detected in input.")

labels_np = np.zeros(len(freq_np), dtype=np.int64)  # temporary labels
print(freq_np.shape, labels_np.shape)

(2, 224, 224, 1) (2,)


In [32]:
x = torch.tensor(freq_np, dtype=torch.float32).permute(0, 3, 1, 2)
y = torch.tensor(labels_np, dtype=torch.long)



In [33]:
class FrequencyCNN(nn.Module):
    def __init__(self, num_classes, emb_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),  # 112
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), # 56
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),# 28
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.embed = nn.Linear(256, emb_dim)      # frequency vector layer
        self.cls = nn.Linear(emb_dim, num_classes)

    def forward(self, x):
        f = self.features(x).flatten(1)           # (B,256)
        z = self.embed(f)                         # (B,emb_dim)
        z = F.normalize(z, p=2, dim=1)           # normalized frequency vector
        logits = self.cls(z)
        return logits, z


In [34]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits, _ = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def extract_frequency_vectors(model, loader, device):
    model.eval()
    vecs, ys = [], []
    for x, y in loader:
        x = x.to(device)
        _, z = model(x)
        vecs.append(z.cpu())
        ys.append(y)
    return torch.cat(vecs), torch.cat(ys)


In [28]:
# Example usage:
# freq_np = your stacked freq data from run_preprocessing_media(...)
# labels_np = np.array([...], dtype=np.int64)

device = "cuda" if torch.cuda.is_available() else "cpu"
train_ds = FrequencyDataset(freq_np, labels_np)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

model = FrequencyCNN(num_classes=len(np.unique(labels_np)), emb_dim=128).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    loss = train_one_epoch(model, train_loader, opt, device)
    print(f"Epoch {epoch+1}: loss={loss:.4f}")

freq_vectors, y_out = extract_frequency_vectors(model, train_loader, device)
print("Frequency vectors shape:", freq_vectors.shape)  # (N, 128)


Epoch 1: loss=0.0000
Epoch 2: loss=0.0000
Epoch 3: loss=0.0000
Epoch 4: loss=0.0000
Epoch 5: loss=0.0000
Epoch 6: loss=0.0000
Epoch 7: loss=0.0000
Epoch 8: loss=0.0000
Epoch 9: loss=0.0000
Epoch 10: loss=0.0000
Frequency vectors shape: torch.Size([2, 128])
